# LEMON Respiration EDA

Run all cells to review the published tables and seven figures, and regenerate the Pearson and Spearman figure. To reanalyse raw recordings, set `RUN_FROM_RAW = True` and point `RAW_DATA` to the respiration TSV/JSON pairs. The default raw path is the local pilot folder. Use the complete dataset path for a full rerun.

In [1]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display

cwd = Path.cwd()
ROOT = next(
    (
        folder for folder in (cwd, *cwd.parents)
        if (folder / "analysis/respiration/scripts/lemon_respiration_eda.py").is_file()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open the notebook from inside the project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RUN_FROM_RAW = False
RAW_DATA = ROOT / "data/lemon/pilot"
CANDIDATES = ROOT / "data/derived/respiration/psq_bold_ecg_candidates.csv"
DEMOGRAPHICS = (
    ROOT / "data/lemon/source/Behavioural_Data_MPILMBB_LEMON"
    / "META_File_IDs_Age_Gender_Education_Drug_Smoke_SKID_LEMON.csv"
)
OUTPUT = ROOT / "data/derived/respiration/notebook"

## Data and outputs

The raw run checks breathing signals and scanner timing, estimates breathing rate by Welch and peak intervals, and compares coverage across groups. It writes new outputs to the notebook results folder. The published tables remain available for the default review.

In [2]:
if RUN_FROM_RAW:
    paths = (RAW_DATA, CANDIDATES, DEMOGRAPHICS)
    missing = [path for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Configure the missing raw inputs: {missing}")
    table_dir = figure_dir = OUTPUT / "raw_run"
    subprocess.run(
        [
            sys.executable, "-m", "analysis.respiration.scripts.lemon_respiration_eda",
            "--data", str(RAW_DATA),
            "--base", str(CANDIDATES),
            "--demographics", str(DEMOGRAPHICS),
            "--output", str(table_dir),
        ],
        cwd=ROOT,
        check=True,
    )
    correlation_figure = figure_dir / "respiration_correlations.png"
else:
    table_dir = ROOT / "data/derived/respiration"             # participant-level tables (not in Git)
    figure_dir = ROOT / "analysis/respiration/figures"
    correlation_figure = OUTPUT / "respiration_correlations.png"

participants = pd.read_csv(table_dir / "participant_eda.csv")
coverage = pd.read_csv((table_dir if RUN_FROM_RAW else ROOT / "analysis/respiration/results") / "cohort_coverage.csv")
windows = pd.read_csv(table_dir / "minute_windows.csv")
recordings = participants.loc[participants["respiration_available"]]
candidates = participants.loc[participants["psq_bold_ecg_candidate"]]
matched = candidates.loc[candidates["respiration_available"]]

## Summary and scan timing

A possible 657-trigger window is a timing candidate. Scan onset needs independent confirmation before physiological correction.

In [3]:
starts = matched["candidate_657_starts"]
print(f"Breathing recordings: {len(recordings)}")
print(f"Candidates with breathing data: {len(matched)}/{len(candidates)}")
print(f"Median Welch rate: {recordings['spectrum_bpm'].median():.2f} breaths/min")
print(f"Median absolute method difference: {recordings['method_abs_difference_bpm'].median():.2f} breaths/min")
print(f"Possible scan starts: {(starts == 1).sum()} unique, {(starts > 1).sum()} multiple, {(starts == 0).sum()} absent")
print(f"One-minute windows: {len(windows)}")
display(coverage)

Breathing recordings: 163
Candidates with breathing data: 156/206
Median Welch rate: 16.53 breaths/min
Median absolute method difference: 0.38 breaths/min
Possible scan starts: 54 unique, 100 multiple, 2 absent
One-minute windows: 2515


,variable,group,n,respiration_n,missing_n,coverage,wilson_low,wilson_high
0,age_group,20–39,142,112,30,0.788732,0.714442,0.847813
1,age_group,55–79,64,44,20,0.687500,0.566077,0.787689
2,age_bin,20-25,72,51,21,0.708333,0.594943,0.800619
3,age_bin,25-30,57,49,8,0.859649,0.746755,0.927127
4,age_bin,30-35,12,11,1,0.916667,0.646120,0.985135
5,age_bin,35-40,1,1,0,1.000000,0.206549,1.000000
6,age_bin,55-60,4,4,0,1.000000,0.510109,1.000000
7,age_bin,60-65,17,13,4,0.764706,0.527382,0.904450
8,age_bin,65-70,21,12,9,0.571429,0.365466,0.755300
9,age_bin,70-75,18,12,6,0.666667,0.437495,0.837212


## Figures

Coverage, trigger timing, signal quality, method agreement, within-recording variation, example traces, sensitivity, and correlations.

In [4]:
if RUN_FROM_RAW:   # show the figures of a fresh run; otherwise the published ones are linked below
    for path in sorted(figure_dir.glob('*.png')):
        display(Image(filename=str(path), width=900))

## Published figures

**respiration_coverage**

![respiration_coverage](figures/respiration_coverage.png)

**respiration_trigger_timing**

![respiration_trigger_timing](figures/respiration_trigger_timing.png)

**respiration_quality**

![respiration_quality](figures/respiration_quality.png)

**respiration_rate_agreement**

![respiration_rate_agreement](figures/respiration_rate_agreement.png)

**respiration_within_recording**

![respiration_within_recording](figures/respiration_within_recording.png)

**respiration_examples**

![respiration_examples](figures/respiration_examples.png)

**respiration_sensitivity**

![respiration_sensitivity](figures/respiration_sensitivity.png)

**respiration_correlations**

![respiration_correlations](figures/respiration_correlations.png)